In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 5.5 Ergodicity: Time Averages versus Ensemble Averages

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume V — Classical Statistical Mechanics",
    number="5.5",
    title="Ergodicity: Time Averages versus Ensemble Averages",
    blurb="The hidden assumption under everything. What a thermometer measures "
    "is a time average along one trajectory; what the microcanonical ensemble "
    "computes is an average over phase space. The ergodic hypothesis says they "
    "agree — and we can watch it hold for a chaotic system, and fail for an "
    "integrable one, with our own integrator.",
    difficulty="advanced",
    estimate="160–200 min",
)

## Notebook overview

[§5.4](microstates-entropy-temperature.ipynb) built the microcanonical ensemble and computed averages by summing over the
equally-probable microstates on an energy surface. It worked — but it rested on a question we
never asked: *why should an average over phase space have anything to do with what an experiment
measures?* A thermometer does not sample microstates; it sits in contact with one system,
following one trajectory through phase space, and reports a **time average** along that
trajectory. The ensemble, by contrast, computes a **phase-space average** over the energy
surface all at once. The claim that these two numbers agree is the **ergodic hypothesis**, and
the entire ensemble method — everything in this volume — stands or falls on it.

This notebook examines that assumption directly, with our own integrator. It is the
statistical-mechanics payoff of Volume II: we lean hard on Hamilton's equations, phase space, and
above all **Liouville's theorem** from [§2.3](../02-classical-mechanics/hamiltonian-phase-flow.ipynb) (Hamiltonian flow), which tells us *which*
measure on the energy surface the dynamics preserves — and so which one the ensemble must use.
But Liouville is not the whole story. It explains why the uniform measure is the natural
equilibrium one; it does *not* guarantee that a single trajectory ever visits all of the surface.
That further property is ergodicity, and whether a system has it turns out to depend on whether
it is **integrable** (it does not) or **chaotic** (it effectively does).

We will see all three faces. The **harmonic oscillator** is ergodic *trivially* — its energy
surface is a one-dimensional curve and the trajectory is that whole curve — and its time-average
position obeys the arcsine law, matching the microcanonical distribution exactly. **Two uncoupled
oscillators** are *integrable*: a second conserved quantity confines the motion to a torus that
cannot fill the energy surface, so the time average depends on initial conditions and ergodicity
fails. The **Hénon–Heiles** system is regular at low energy and *chaotic* near its escape energy,
with a positive Lyapunov exponent; there a single trajectory wanders over the surface and the
time average converges back to the ensemble average. The lesson is the one that licenses
statistical mechanics: realistic many-body systems are chaotic enough to be effectively ergodic —
and ergodicity is an assumption, demonstrable and breakable, not a theorem.

> **How to read the checks.** Each exercise closes with a `validate` call against an independent
> fact: a time average that settles; a phase-space area conserved under a nonlinear flow
> (Liouville); the oscillator's arcsine law matching the microcanonical distribution; a second
> invariant pinning an integrable system to a torus; energy conserved by the integrator; a
> Poincaré section that is curves at low energy and a filled area at high energy; a Lyapunov
> exponent positive in the chaotic regime and near zero in the regular one; and the chaotic time
> average converging to the ensemble average. A ✓ is strong evidence; a ✗ is a prompt to *locate
> the discrepancy*, not a verdict.
>
> **Scope.** Why the ensemble method is allowed — the time-average/ensemble-average bridge.
> Liouville's theorem itself is developed in [§2.3](../02-classical-mechanics/hamiltonian-phase-flow.ipynb); the microcanonical ensemble in [§5.4](microstates-entropy-temperature.ipynb).
> See Goldstein, *Classical Mechanics*; Pathria & Beale, *Statistical Mechanics*; Tuckerman,
> *Statistical Mechanics: Theory and Molecular Simulation*; and Hénon & Heiles (1964).

## Theory in brief

### Two kinds of average

Let $A$ be an observable — a function of the phase-space point $(\mathbf q,\mathbf p)$. Its
**time average** along the trajectory starting at $(\mathbf q_0,\mathbf p_0)$ is what any
measurement or molecular-dynamics run reports,

```{math}
:label: eq-two-averages
\bar A=\lim_{T\to\infty}\frac1T\int_0^T A\big(\mathbf q(t),\mathbf p(t)\big)\,dt ,
\qquad
\langle A\rangle=\int A\,d\mu ,
```

while its **ensemble average** $\langle A\rangle$ integrates $A$ over the microcanonical measure
$\mu$ on the energy surface $H=E$ — the equally-probable microstates of [§5.4](microstates-entropy-temperature.ipynb). The **ergodic
hypothesis** asserts $\bar A=\langle A\rangle$ for almost every starting point. Statistical
mechanics computes $\langle A\rangle$ but predicts the measured $\bar A$; the whole method is the
bet that these agree.

### Liouville's theorem grounds the measure

Why the energy surface, and why a *uniform* measure on it? Because of Liouville's theorem
([§2.3](../02-classical-mechanics/hamiltonian-phase-flow.ipynb)): under Hamiltonian flow the phase-space density is incompressible — a blob of
initial conditions is sheared and folded but its phase-space **volume** is conserved,

```{math}
:label: eq-erg-liouville
\frac{d}{dt}\!\int_{\text{blob}}\!d\mathbf q\,d\mathbf p=0 .
```

A uniform density is therefore **stationary**: spread states evenly over the energy surface and
the flow keeps them evenly spread. That is what singles out the microcanonical measure as the
equilibrium one — it is the measure the dynamics preserves. But Liouville says nothing about a
*single* trajectory: it could still be trapped on a sub-region. Ergodicity is the extra demand
that one trajectory explores the whole surface.

### The cleanest case: the harmonic oscillator

For one oscillator $H=p^2/2m+\tfrac12 m\omega^2x^2$, the energy surface is a single ellipse in
$(x,p)$, and the trajectory *is* that whole ellipse — so the system is ergodic for free. The
time-average position density is the **arcsine law** (more time is spent near the slow turning
points),

```{math}
:label: eq-arcsine
p(x)=\frac{1}{\pi\sqrt{A^2-x^2}},\qquad
F(x)=\tfrac12+\frac{1}{\pi}\arcsin\!\frac{x}{A},
```

with amplitude $A$, and it equals the microcanonical distribution exactly. Every observable
agrees both ways: $\langle x^2\rangle=A^2/2=E/m\omega^2$.

### When ergodicity fails: integrable systems

A system with as many independent conserved quantities as degrees of freedom is **integrable**,
and its motion lies on a **torus** in phase space (the KAM picture) of lower dimension than the
energy surface,

```{math}
:label: eq-integrable
\text{two uncoupled oscillators conserve } E_x \text{ and } E_y \text{ separately} ,
```

so the trajectory cannot fill the surface; the time average then depends on *which* torus — on
the initial conditions — and differs from the microcanonical average. Integrable means
non-ergodic.

### When ergodicity (approximately) holds: chaos

A **chaotic** system has no extra conserved quantity and shows sensitive dependence on initial
conditions — nearby trajectories separate exponentially, $\delta(t)\sim\delta_0\,e^{\lambda t}$,

```{math}
:label: eq-chaos
\lambda=\lim_{t\to\infty}\frac1t\ln\frac{\lvert\delta(t)\rvert}{\lvert\delta_0\rvert}>0
\quad(\text{a positive Lyapunov exponent}) ,
```

and a single trajectory wanders over (most of) the energy surface, so the time average returns to
the ensemble average. The **Hénon–Heiles** system $H=\tfrac12(p_x^2+p_y^2)+\tfrac12(x^2+y^2)+
x^2y-\tfrac13 y^3$ is regular at low energy (tori) and chaotic near its escape energy $E=1/6$.
Its **Poincaré section** — the crossings of a plane — shows the contrast directly: nested closed
curves versus a scattered sea. Realistic many-body systems are overwhelmingly chaotic, hence
effectively ergodic — which is *why* the ensembles describe real matter. But ergodicity can
**break** (a glass; a magnet frozen in one sector below $T_c$), the deep idea the Ising capstone
returns to.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from scipy.integrate import solve_ivp

from ecp import draw, validate
from ecp.animate import show

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT

# Common high-accuracy integration settings, defined here and reused.
RTOL, ATOL, METHOD = 1e-10, 1e-12, "DOP853"


def oscillator_rhs(t, s, omega=1.0):
    """Hamilton's equations for a unit-mass harmonic oscillator H = p^2/2 + (1/2)ω^2 x^2.

    Parameters
    ----------
    t : float
        Time (unused; the system is autonomous, but ``solve_ivp`` passes it).
    s : sequence of float
        Phase-space point ``(x, p)``.
    omega : float, optional
        Angular frequency (default ``1.0``).

    Returns
    -------
    list of float
        The time derivatives ``(x_dot, p_dot) = (p, -omega**2 x)``.
    """
    x, p = s
    return [p, -(omega**2) * x]


def pendulum_rhs(t, s):
    """Hamilton's equations for a pendulum H = p^2/2 − cos q (a genuinely nonlinear flow).

    Parameters
    ----------
    t : float
        Time (unused; autonomous).
    s : sequence of float
        Phase-space point ``(q, p)``.

    Returns
    -------
    list of float
        ``(q_dot, p_dot) = (p, -sin q)``.
    """
    q, p = s
    return [p, -np.sin(q)]


def two_oscillator_rhs(t, s, wx=1.0, wy=np.sqrt(2.0)):
    """Hamilton's equations for two uncoupled oscillators (an integrable system).

    With an irrational frequency ratio ``wy/wx`` the motion is quasi-periodic on a 2-torus; each
    oscillator's energy is separately conserved — the second invariant that makes the system
    integrable and therefore non-ergodic.

    Parameters
    ----------
    t : float
        Time (unused; autonomous).
    s : sequence of float
        Phase-space point ``(x, px, y, py)``.
    wx, wy : float, optional
        The two frequencies (default ``1`` and ``sqrt(2)``).

    Returns
    -------
    list of float
        ``(x_dot, px_dot, y_dot, py_dot)``.
    """
    x, px, y, py = s
    return [px, -(wx**2) * x, py, -(wy**2) * y]


def henon_heiles_rhs(t, s):
    """Hamilton's equations for the Hénon–Heiles system H = (1/2)(px^2+py^2) + (1/2)(x^2+y^2) + x^2·y − y^3/3.

    A 2-degree-of-freedom system that is regular (tori) at low energy and chaotic near its escape
    energy E = 1/6. The forces are −∂H/∂x = −(x + 2xy) and −∂H/∂y = −(y + x^2 − y^2).

    Parameters
    ----------
    t : float
        Time (unused; autonomous).
    s : sequence of float
        Phase-space point ``(x, px, y, py)``.

    Returns
    -------
    list of float
        ``(x_dot, px_dot, y_dot, py_dot)``.
    """
    x, px, y, py = s
    return [px, -(x + 2 * x * y), py, -(y + x * x - y * y)]


def henon_heiles_energy(s):
    """The Hénon–Heiles energy H at a phase-space point (used as an integrator check).

    Parameters
    ----------
    s : numpy.ndarray
        Phase-space point(s) ``(x, px, y, py)`` (each may be an array over time).

    Returns
    -------
    float or numpy.ndarray
        The energy H = (1/2)(px^2+py^2) + (1/2)(x^2+y^2) + x^2·y − y^3/3.
    """
    x, px, y, py = np.asarray(s)
    return 0.5 * (px * px + py * py) + 0.5 * (x * x + y * y) + x * x * y - y**3 / 3.0


def henon_heiles_ic(E, y, py, x=0.0):
    """A Hénon–Heiles initial condition on the energy surface H = E, on the section plane x = 0.

    Solves H = E for px ≥ 0 given (x=0, y, py), so the trajectory starts on the
    Poincaré plane heading in the +x direction.

    Parameters
    ----------
    E : float
        Target energy (bound motion requires E < 1/6).
    y, py : float
        The chosen y and py on the section plane.
    x : float, optional
        Position (default ``0.0``, the section plane).

    Returns
    -------
    list of float
        ``(x, px, y, py)`` with px = √(2E − py^2 − y^2 + (2/3)y^3) (valid at x = 0).
    """
    px2 = 2 * E - py * py - y * y + (2.0 / 3.0) * y**3
    return [x, np.sqrt(max(px2, 0.0)), y, py]


def polygon_area(xy):
    """The area of a simple polygon via the shoelace formula.

    Parameters
    ----------
    xy : numpy.ndarray, shape (n, 2)
        Ordered vertices of the polygon.

    Returns
    -------
    float
        The enclosed area (1/2)|Σ_i (x_i y_{i+1} − x_{i+1} y_i)|.
    """
    x, y = xy[:, 0], xy[:, 1]
    return 0.5 * np.abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))


def poincare_section(rhs, s0, t_max, max_step=0.5):
    """Compute a Poincaré section: the crossings of the plane x = 0 with px > 0.

    Integrates the flow and records (y, py) each time the trajectory pierces x = 0 moving in
    the +x direction, using ``scipy.integrate.solve_ivp`` event detection.

    Parameters
    ----------
    rhs : callable
        The Hamiltonian right-hand side ``rhs(t, s)`` with state ``(x, px, y, py)``.
    s0 : sequence of float
        Initial phase-space point.
    t_max : float
        Integration time (longer gives more section points).
    max_step : float, optional
        Cap on the integrator step so no crossing is skipped (default ``0.5``).

    Returns
    -------
    tuple of numpy.ndarray
        Arrays ``(y, py)`` of the section crossings.
    """

    def event(t, s):
        return s[0]  # x = 0

    event.direction = 1.0  # only px > 0 (x increasing)
    sol = solve_ivp(
        rhs,
        [0, t_max],
        s0,
        method=METHOD,
        rtol=RTOL,
        atol=ATOL,
        events=event,
        max_step=max_step,
    )
    ev = sol.y_events[0]
    if len(ev) == 0:
        return np.array([]), np.array([])
    return ev[:, 2], ev[:, 3]


def lyapunov_exponent(rhs, s0, t_total=1500.0, tau=5.0, d0=1e-8):
    """Estimate the largest Lyapunov exponent by Benettin's renormalization method.

    Integrates a reference trajectory and a shadow started a tiny distance ``d0`` away; every
    ``tau`` the separation is measured (``numpy.linalg.norm``), its log-growth accumulated, and the
    shadow rescaled back to ``d0`` along the separation direction so it never saturates. The mean
    exponential rate is λ.

    Parameters
    ----------
    rhs : callable
        Hamiltonian right-hand side.
    s0 : sequence of float
        Initial condition for the reference trajectory.
    t_total : float, optional
        Total integration time (default ``1500``).
    tau : float, optional
        Renormalization interval (default ``5``; must be short enough to stay in the linear regime).
    d0 : float, optional
        Reference separation (default ``1e-8``).

    Returns
    -------
    float
        The estimated largest Lyapunov exponent λ.
    """
    s = np.array(s0, dtype=float)
    shadow = s + d0 * np.array([1.0, 0.0, 0.0, 0.0])
    n_steps = int(t_total / tau)
    total = 0.0
    for _ in range(n_steps):
        s = solve_ivp(rhs, [0, tau], s, method=METHOD, rtol=RTOL, atol=ATOL).y[:, -1]
        shadow = solve_ivp(
            rhs, [0, tau], shadow, method=METHOD, rtol=RTOL, atol=ATOL
        ).y[:, -1]
        d = np.linalg.norm(shadow - s)
        total += np.log(d / d0)
        shadow = s + (shadow - s) * (d0 / d)  # renormalize back to d0
    return total / (n_steps * tau)

## Exercise 1 — Two kinds of average

Start with the question the whole notebook answers, posed concretely. Take the harmonic
oscillator, integrate one trajectory, and watch a **time average** build up: at each moment form
the running average $\frac1T\int_0^T x^2\,dt$ of the squared position along the path
({numref}`fig-erg-thesis`). It settles quickly to a definite number — a measurement would report
exactly this. The ensemble, by contrast, would compute $\langle x^2\rangle$ by integrating over
the energy surface all at once {eq}`eq-two-averages`, never following any trajectory at all. Are
these the same number? For this system we will prove yes (Exercise 3); for an integrable system
the answer is no (Exercise 4). The entire validity of statistical mechanics is the claim that,
for the systems that matter, they agree.

**This exercise (worked).** Integrate the oscillator with `scipy.integrate.solve_ivp`
(`DOP853`, `rtol=1e-10`, `atol=1e-12`), form the running time average of $x^2$ with
`numpy.cumsum`, and confirm it *converges* — the tail is flat. Whether that limit equals the
phase-space average is the question we defer to Exercise 3.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    tail_spread < 1e-2,
    "a time average along the trajectory settles to a definite value",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — Liouville's theorem: the measure the flow preserves

Before comparing the two averages we must say *which* ensemble measure is the right one, and
Liouville's theorem from [§2.3](../02-classical-mechanics/hamiltonian-phase-flow.ipynb) answers it. Hamiltonian flow is **incompressible**: take a
blob of initial conditions and let every point flow, and the blob shears, stretches, and curls,
but its phase-space **area** never changes {eq}`eq-erg-liouville` ({numref}`fig-erg-liouville`). The
consequence is decisive: a measure that is *uniform* over the energy surface is stationary —
evenly spread states stay evenly spread — so the uniform (microcanonical) measure is the one the
dynamics preserves, the natural equilibrium ensemble. This is why [§5.4](microstates-entropy-temperature.ipynb) weighted every microstate
equally. But notice what Liouville does *not* say: it constrains the whole blob, never a single
point. One trajectory could still be trapped on part of the surface forever. That gap is exactly
where ergodicity lives.

**This exercise (worked).** Evolve a ring of initial conditions under the genuinely nonlinear
**pendulum** flow $H=p^2/2-\cos q$ (`scipy.integrate.solve_ivp`), and compute the ring's enclosed
area with the shoelace formula at several times. The ring distorts dramatically; its area is
conserved to three digits — Liouville's theorem, watched directly.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    areas[-1],
    areas[0],
    "the phase-space area of a blob is conserved under Hamiltonian flow (Liouville)",
    rtol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — The oscillator is ergodic: time average = ensemble average

Now the cleanest possible test of the ergodic equality. For one oscillator the energy surface
$H=E$ is a single closed curve in $(x,p)$, and the trajectory traces that *entire* curve, over
and over — so a single trajectory manifestly visits the whole surface, and the system is ergodic
trivially. We can therefore predict the time-average distribution of position analytically: the
particle moves slowest near the turning points $\pm A$, so it spends most of its time there, and
the density is the **arcsine law** $p(x)=1/\pi\sqrt{A^2-x^2}$ {eq}`eq-arcsine`
({numref}`fig-erg-arcsine`). This is *also* the microcanonical distribution — the fraction of the
energy surface at each $x$ — so the two agree by construction. We confirm it two ways: the
empirical and analytic cumulative distributions coincide (comparing CDFs sidesteps the binning
singularity at $\pm A$), and $\langle x^2\rangle$ comes out the same as a time average and as the
ensemble value $E/m\omega^2$.

**This exercise (worked).** From a long oscillator trajectory, sort the sampled positions into an
empirical CDF (`numpy.sort`), compare to the arcsine CDF $F(x)=\tfrac12+\arcsin(x/A)/\pi$, and
confirm the maximum difference is tiny; then check that the time-averaged $\langle x^2\rangle$
equals the ensemble value $A^2/2=E/m\omega^2$.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    max_cdf_diff,
    0.0,
    "the oscillator's time-average position law matches the microcanonical arcsine (via the CDF)",
    atol=1e-2,
)
validate.close(
    x2_time,
    x2_ensemble,
    "⟨x²⟩ is the same as a time average and as an ensemble average",
    rtol=1e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — An integrable system is not ergodic: motion on a torus

Here the ergodic equality *fails*, and seeing exactly why is the point. Take two uncoupled
oscillators with an irrational frequency ratio. Because they never exchange energy, each
oscillator's energy $E_x$ and $E_y$ is conserved *separately* — that is a second invariant beyond
the total energy {eq}`eq-integrable`. The total energy alone would allow the system to roam a
three-dimensional surface, but the two separate conservation laws pin the motion to a
two-dimensional **torus**: the trajectory winds densely around it (the Lissajous figure) yet
never leaves it ({numref}`fig-erg-torus`). On the $(E_x,E_y)$ plane the motion sits at a single
point for all time — it never explores any other division of the energy. So the time average of,
say, $x^2$ is fixed by *which* torus we started on; change the initial amplitudes and it changes.
The time average depends on initial conditions, and there is no single ensemble value it equals.
Integrable means non-ergodic.

**This exercise (worked).** Integrate the two-oscillator system, confirm $E_x$ and $E_y$ are each
conserved (their ranges over the whole trajectory are $\sim10^{-11}$, a second invariant), and
show the motion is a bounded torus that never fills the energy surface.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    [range_Ex, range_Ey],
    [0.0, 0.0],
    "each oscillator's energy is separately conserved — a second invariant confines the motion to a torus (non-ergodic)",
    atol=1e-6,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — The Hénon–Heiles system: regular and chaotic regimes

To find genuine ergodicity we need a system that is *not* integrable, and the **Hénon–Heiles**
model is the classic. It describes a star moving in the plane of a galaxy,
$H=\tfrac12(p_x^2+p_y^2)+\tfrac12(x^2+y^2)+x^2y-\tfrac13 y^3$ — two oscillators coupled by the
cubic terms — and its potential is a triangular well that confines bound motion for energies
below the escape value $E=1/6$ ({numref}`fig-erg-potential`). The coupling destroys the second
invariant: there is no $E_x$, no $E_y$, only the total energy. At *low* energy the system stays
nearly regular (the tori survive, deformed); near the escape energy it becomes *chaotic*. Before
we can study either regime we need an integrator we trust, so the first check is that energy is
conserved — for a 2-degree-of-freedom nonlinear system, holding $H$ fixed to $\sim10^{-10}$ over
hundreds of time units is the certificate that the dynamics, not numerical drift, is what we are
seeing.

**This exercise (worked).** Integrate Hénon–Heiles at low energy $E=0.04$ and high energy
$E=0.16$ with `scipy.integrate.solve_ivp` (`DOP853`), and confirm the energy
`henon_heiles_energy` is conserved to high accuracy in both runs.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    max(drifts.values()),
    0.0,
    "the high-accuracy integration conserves the Hénon–Heiles energy",
    atol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — Poincaré sections: tori versus the chaotic sea

The Poincaré section turns a tangled four-dimensional flow into a picture you can read at a
glance, and it is the iconic image of the integrable-to-chaotic transition. Each time a
trajectory pierces the plane $x=0$ moving in the $+x$ direction, we drop a dot at its $(y,p_y)$
{eq}`eq-integrable`,{eq}`eq-chaos`. A trajectory on a surviving torus pierces the plane on a
smooth closed loop — its dots trace a **curve**. A chaotic trajectory has no such invariant
surface to live on, so its dots scatter and **fill an area** ({numref}`fig-erg-poincare`). At
low energy ($E=0.04$) every trajectory gives nested closed curves: the motion is regular. Near
the escape energy ($E=0.16$) a single trajectory's points spray across a two-dimensional sea:
the motion is chaotic, and that one trajectory is already exploring most of the accessible
surface — the geometric signature of (approximate) ergodicity.

**This exercise (worked).** Compute Poincaré sections (`poincare_section`, using `solve_ivp`
event detection) for several initial conditions at each energy, and quantify the contrast with a
grid-occupancy measure (`numpy.histogram2d`): the chaotic section fills several times more cells
than the regular one.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    occ_hi > 2.5 * occ_lo and occ_lo < 0.2,
    "the Poincaré section is thin curves (tori) at low energy and a space-filling sea (chaos) at high energy",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — The Lyapunov exponent: measuring chaos

What *makes* a single chaotic trajectory fill the surface? Sensitive dependence on initial
conditions: two trajectories that start a hair apart separate exponentially, $\lvert\delta(t)
\rvert\sim\lvert\delta_0\rvert\,e^{\lambda t}$, and the rate $\lambda$ is the **largest Lyapunov
exponent** {eq}`eq-chaos`. A positive $\lambda$ is the quantitative fingerprint of chaos — and it
is precisely the mechanism of effective ergodicity, because exponential stretching is what folds
one trajectory through every corner of the accessible surface. We measure $\lambda$ by Benettin's
trick: run a reference trajectory and a shadow a tiny distance away, and every short interval
record how much their separation grew and then rescale the shadow back, so it tracks the local
stretching forever without saturating ({numref}`fig-erg-lyapunov`). At high energy $\lambda$ is
clearly positive; at low energy it decays toward zero (regular motion separates only as a power
of $t$, so the running estimate drifts down as $\sim\ln t/t$).

**This exercise (worked).** Estimate $\lambda$ with `lyapunov_exponent` (using `numpy.linalg.norm`
for the separation) at $E=0.16$ and $E=0.04$, and confirm it is positive in the chaotic regime
and near zero in the regular one.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    lam_hi > 0.04 and lam_lo < 0.025,
    "the Lyapunov exponent is positive in the chaotic regime and near zero in the regular regime",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 8 — Time average meets ensemble average in the chaotic regime

Now we close the loop the whole notebook has been building toward. In the chaotic regime a single
trajectory explores (most of) the energy surface, so its **time average** of an observable should
converge to the **ensemble average** over that surface {eq}`eq-two-averages`,{eq}`eq-chaos` — the
ergodic equality, restored by chaos. We test it on $\langle y^2\rangle$ at $E=0.16$. The ensemble
average is computed directly from the microcanonical measure: by Liouville the measure is uniform
in the accessible configuration region (with the momentum direction uniform), so $\langle y^2
\rangle_{\rm ens}$ is just the area-weighted average of $y^2$ over $\{V(x,y)\le E\}$. The time
average comes from long chaotic trajectories. They agree to a few percent
({numref}`fig-erg-ergodic`) — and crucially, *different* chaotic starting points give the *same*
time average, the true mark of ergodicity. Contrast the integrable two-oscillator system, whose
time average is set by the initial torus and refuses to settle on any common value. Realistic
matter is chaotic; that is why the ensemble averages of [§5.4](microstates-entropy-temperature.ipynb) predict what is measured.

**This exercise (student).** Estimate $\langle y^2\rangle_{\rm ens}$ by Monte Carlo sampling the
accessible region (`numpy.random.default_rng`), average $y^2$ over several long chaotic
trajectories, and confirm they agree. Then show the integrable system's time average is instead
initial-condition-dependent — ergodicity holds in one case and fails in the other.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    y2_time,
    y2_ensemble,
    "in the chaotic regime the time average converges to the ensemble average — ergodicity, restored by chaos",
    rtol=0.1,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 9 — Why the ensembles are allowed

Stand back and see what we have established. Statistical mechanics computes **ensemble averages**
— integrals over the energy surface — but every measurement is a **time average** along one
trajectory. The bridge between them is **ergodicity**, and we have tested it from every side.
Liouville's theorem ([§2.3](../02-classical-mechanics/hamiltonian-phase-flow.ipynb)) told us *which* measure the dynamics preserves, singling out
the uniform microcanonical measure of [§5.4](microstates-entropy-temperature.ipynb) as the equilibrium one. The harmonic oscillator showed
the time and ensemble averages can be *exactly* equal — its trajectory is the whole energy curve.
Two uncoupled oscillators showed the equality can *fail* — a second invariant traps the motion on
a torus, and the time average remembers its initial conditions. The Hénon–Heiles system showed
what *rescues* it — chaos, a positive Lyapunov exponent, a single trajectory spilling across the
whole surface so the time average returns to the ensemble value. Realistic many-body systems are
overwhelmingly chaotic, hence effectively ergodic, and that — not a theorem — is why the
microcanonical ensemble describes real matter. The ensembles to come rest on the same foundation.

**This exercise (synthesis).** No new computation: the argument is the result. We compute over all
of phase space and trust that one trajectory, given time, visits it all. For a chaotic system it
does — we measured it — and that trust is the foundation under everything that follows. Where it
breaks, so does the ensemble description, which is the warning the Ising capstone will take up as
**ergodicity breaking**.

## Notebook summary

Statistical mechanics computes phase-space averages but predicts time averages; this notebook
examined the assumption — ergodicity — that lets it, building each step on Volume II's Hamiltonian
dynamics and Liouville's theorem.

- **Two kinds of average** {eq}`eq-two-averages`: the time average $\bar A$ along a trajectory
  (what is measured) versus the ensemble average $\langle A\rangle$ over the energy surface (what
  is computed); the ergodic hypothesis is $\bar A=\langle A\rangle$.
- **Liouville's theorem** {eq}`eq-erg-liouville`: Hamiltonian flow is incompressible (a pendulum blob
  conserves its area to three digits), so the uniform measure on the energy surface is stationary —
  the equilibrium measure of [§5.4](microstates-entropy-temperature.ipynb). But it does not by itself guarantee ergodicity.
- **The oscillator** {eq}`eq-arcsine`: trivially ergodic; the time-average position is the
  arcsine law, matching the microcanonical CDF to $\sim10^{-6}$, and $\langle x^2\rangle=E/m\omega^2$
  both ways.
- **Integrable = non-ergodic** {eq}`eq-integrable`: two uncoupled oscillators conserve $E_x,E_y$
  separately (ranges $\sim10^{-11}$), confining the motion to a torus; the time average depends on
  initial conditions.
- **Chaos restores ergodicity** {eq}`eq-chaos`: Hénon–Heiles is regular at $E=0.04$ and chaotic
  at $E=0.16$ (energy conserved to $\sim10^{-10}$); its Poincaré section turns from nested curves
  into a space-filling sea, its Lyapunov exponent from $\approx0$ to $\approx0.10$, and the
  chaotic time average of $\langle y^2\rangle$ converges to the ensemble average (a few percent).

The ensemble method is licensed because realistic systems are chaotic enough to be effectively
ergodic — an assumption, demonstrable and breakable, not a theorem.

## Outlook

- **Mixing and the approach to equilibrium.** A stronger property than ergodicity — not just that
  a trajectory visits everywhere, but that any initial distribution relaxes to uniform — and the
  timescales on which a system forgets where it started.
- **Molecular dynamics.** The practical face of the time average: a single long trajectory,
  integrated exactly as we did here, standing in for the ensemble — the bridge to the simulation
  methods of this volume and to materials modelling.
- **The other derivatives of entropy ([§5.6](ideal-gas-fundamental-relation.ipynb)).** With the ensemble method justified, pressure and
  chemical potential as $\partial S/\partial V$ and $\partial S/\partial N$, and the fundamental
  thermodynamic relation — the rest of thermodynamics from the same $S=k\ln\Omega$.
- **The canonical ensemble.** Summing the Boltzmann factor of [§5.4](microstates-entropy-temperature.ipynb) over a system's states into the
  partition function $Z$, from which $\ln Z$ generates all thermodynamics — developed later in the
  volume, and resting on the same ergodic foundation examined here.
- **Ergodicity breaking (the Ising capstone, [§5.10](ising-emergence-universality.ipynb)).** When a system gets trapped in part of phase
  space — a glass, or a ferromagnet frozen in one magnetization sector below $T_c$ — the time and
  ensemble averages part ways, and the ensemble description must be handled with care.
- **Cross-reference** [§2.3](../02-classical-mechanics/hamiltonian-phase-flow.ipynb) (Hamiltonian flow and Liouville's theorem) and [§5.4](microstates-entropy-temperature.ipynb) (the
  microcanonical ensemble).

In [ ]:
from ecp.style import footer

footer()